# 03 · Retrieval — Query 技巧（Rewrite / Multi-Query / HyDE）

目标：
- 复用课件里的“Query 改写”思路
- 在同一个 Chroma collection 上对比：
  - baseline（原始 query）
  - query rewrite（LLM 改写）
  - multi-query（生成多条查询，融合召回）
  - HyDE（先生成假设答案，再用它去检索）

> 依赖：上一节已写入 `data/chroma` + 设置 `OPENAI_API_KEY`。


In [4]:
pip install langchain_openai langchain_community

  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_classic-1.0.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached sqlalchemy-2.0.48-cp311-cp311-macosx_11_0_arm64.whl.metadata (9.5 kB)
  Using cached dataclasses_json-0.6.7-py3-none-any.whl.metadata (25 kB)
  Using cached pydantic_settings-2.13.1-py3-none-any.whl.metadata (3.4 kB)
  Using cached httpx_sse-0.4.3-py3-none-any.whl.metadata (9.7 kB)
  Using cached marshmallow-3.26.2-py3-none-any.whl.metadata (7.3 kB)
  Using cached typing_inspect-0.9.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached langchain_text_splitters-1.1.1-py3-none-any.whl.metadata (3.3 kB)
  Using cached mypy_extensions-1.1.0-py3-none-any.whl.metadata (1.1 kB)
Using cached langchain_community-0.4.1-py3-none-any.whl (2.5 MB)
Using cached dataclasses_json-0.6.7-py3-none-any.whl (28 kB)
Using cached httpx_sse-0.4.3-py3-none-any.whl (9.0 kB)
Using cached langchain_classic-1.0.2-py3-none-any.whl (1.0 MB)
Using cached la

In [1]:
from __future__ import annotations

import os
from pathlib import Path

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma


def resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()

    for candidate in (cwd, cwd.parent):
        if (candidate / "data").exists() and (candidate / "notebooks").exists():
            return candidate

    raise FileNotFoundError("未找到项目根目录，请从 RAG_project 根目录或 notebooks 目录运行本 notebook。")



def load_project_env(project_root: Path) -> Path | None:
    for env_path in (project_root / ".env", project_root.parent / ".env"):
        if env_path.exists():
            load_dotenv(env_path, override=True)
            return env_path
    return None


PROJECT_ROOT = resolve_project_root()
ENV_FILE = load_project_env(PROJECT_ROOT)
CHROMA_DIR = PROJECT_ROOT / "data/chroma"
COLLECTION = "autel_annual_report_2024"

openai_api_key = os.getenv("OPENAI_API_KEY")
openai_base_url = os.getenv("OPENAI_BASE_URL")
embed_model =  os.getenv("EMBED_MODEL") # for Openrouter "qwen/qwen3-embedding-4b"
chat_model = os.getenv("CHAT_MODEL") 

assert CHROMA_DIR.exists(), f"找不到 Chroma 目录：{CHROMA_DIR.resolve()}（先跑 01_data_02_chunk_ingest.ipynb）"
assert openai_api_key, f"未加载 OPENAI_API_KEY（检查 {ENV_FILE or PROJECT_ROOT.parent / '.env'}）"

SECTION_COLLECTION = "autel_annual_report_2024_sections"

client_kwargs = {
    "api_key": openai_api_key,
    "base_url": openai_base_url,
}
print(embed_model)
emb = OpenAIEmbeddings(model=embed_model, **client_kwargs)

import chromadb
_chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))

vs = Chroma(collection_name=COLLECTION, embedding_function=emb, client=_chroma_client)
section_vs = Chroma(collection_name=SECTION_COLLECTION, embedding_function=emb, client=_chroma_client)

llm = ChatOpenAI(model=chat_model, temperature=0, **client_kwargs)
print("ready — chunk collection:", COLLECTION, "| section collection:", SECTION_COLLECTION)
print("embed:", embed_model, "| chat:", chat_model, "| env:", ENV_FILE)


/opt/anaconda3/envs/voc/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.1.0)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
/opt/anaconda3/envs/voc/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Qwen/Qwen3-Embedding-8B
ready — chunk collection: autel_annual_report_2024 | section collection: autel_annual_report_2024_sections
embed: Qwen/Qwen3-Embedding-8B | chat: deepseek-ai/DeepSeek-V3.2 | env: /Users/mengbai/Documents/AI-training/.env


/var/folders/hb/4k5shxzs4c7by7lm5s0mmgrw0000gn/T/ipykernel_75476/1367411917.py:55: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vs = Chroma(collection_name=COLLECTION, embedding_function=emb, client=_chroma_client)


### Baseline

In [2]:
# baseline 检索


QUESTION = "公司在海外市场有哪些布局？"

hits = vs.similarity_search(QUESTION, k=10)
for i, d in enumerate(hits, 1):
    print(f"[{i}]", d.metadata)
    print(d.page_content.replace("\n", " "))
    print()


[1] {'source': '道通24年年报', 'file_path': 'paddleocr_vl/autel_annual_report_2024.md', 'parse_source': 'paddleocr_vl', 'h1': '母公司资产负债表', 'section_title': '2、持续经营', 'content_level': 'chunk', 'section_in_doc': 295, 'chunk_in_doc': 394, 'source_doc_count': 306, 'doc_id': '道通24年年报__full', 'doc_group': 'full_document', 'h3': '2、持续经营', 'chunk_in_section': 0, 'page_start': 1, 'page_end': 306, 'section_id': '道通24年年报__full::section_295', 'chunk_id': 394, 'h2': '四、财务报表的编制基础'}
√适用 □不适用   本公司不存在导致对报告期末起 12 个月内的持续经营能力产生重大疑虑的事项或情况。

[2] {'doc_id': '道通24年年报__full', 'page_start': 1, 'file_path': 'paddleocr_vl/autel_annual_report_2024.md', 'section_in_doc': 585, 'h1': '母公司资产负债表', 'content_level': 'chunk', 'chunk_in_doc': 718, 'source': '道通24年年报', 'page_end': 306, 'parse_source': 'paddleocr_vl', 'chunk_in_section': 0, 'chunk_id': 718, 'source_doc_count': 306, 'section_id': '道通24年年报__full::section_585', 'doc_group': 'full_document', 'h2': '2、本企业的子公司情况', 'section_title': '2、本企业的子公司情况'}
√适用 □不适用   本企业子公司的情况详见本

In [20]:
### Query Rewrite

In [3]:
# 1) Query Rewrite：根据问题主题选择更贴近年报原文标题的检索锚点

from langchain_core.prompts import ChatPromptTemplate

rewrite_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "你是面向中文年报检索的 Query Rewrite 模块。"
            "请先在心里识别用户问题主题，再输出 1 条更适合向量检索的中文 query。\n"
            "原则：\n"
            "1. 保留公司名、年份和核心问题，不要扩大或改变问题范围；\n"
            "2. 优先使用年报中可能真实出现的章节名、小节名、关键词锚点，而不是泛泛改写；\n"
            "3. 如果问题是‘核心竞争力/竞争优势/技术壁垒/护城河/领先性’，优先考虑这类锚点：报告期内核心竞争力分析、核心竞争力分析、技术创新优势、产品及解决方案优势、全球化本地化营销服务体系优势、全球化本地化产能及供应链优势、人才与团队优势；\n"
            "4. 如果问题是其他主题，也按同样思路改写成更贴近年报标题的 query；\n"
            "5. 只输出 1 行 query，不要解释，不要回答问题。",
        ),
        ("human", "原始问题：{q}"),
    ]
)

rewritten = llm.invoke(rewrite_prompt.format_messages(q=QUESTION)).content.strip()
print("rewritten:\n", rewritten)

hits = vs.similarity_search(rewritten, k=10)
for i, d in enumerate(hits, 1):
    print(f"[{i}]", d.metadata)
    print(d.page_content[:260].replace("\n", " "))
    print()


rewritten:
 海外市场布局
[1] {'section_id': '道通24年年报__full::section_45', 'h1': '2024 年度报告', 'doc_id': '道通24年年报__full', 'page_end': 306, 'source_doc_count': 306, 'section_in_doc': 45, 'doc_group': 'full_document', 'chunk_id': 61, 'section_title': '3、报告期内新技术、新产业、新业态、新模式的发展情况和未来发展趋势', 'parse_source': 'paddleocr_vl', 'h2': '3、报告期内新技术、新产业、新业态、新模式的发展情况和未来发展趋势', 'chunk_in_doc': 61, 'source': '道通24年年报', 'file_path': 'paddleocr_vl/autel_annual_report_2024.md', 'page_start': 1, 'chunk_in_section': 0, 'content_level': 'chunk'}
#### (1) 汽车综合诊断及检测行业   根据 Frost&Sullivan 的报告，DaaS（诊断即服务）正成为未来的关键发展趋势，DaaS 是一种通过订阅模式为汽车维修店、经销商等提供车辆诊断工具、远程技术支持和校准设备等资源的服务模式。到 2030 年，DaaS 业务模式预计将产生超过 32.8 亿美元的收入，2023-2030 年的复合年增长率为 12.7%，其中，车辆维修/保养、车辆碰撞维修、电动汽车电池诊断、车辆评估诊断等细分市场都将呈现不同程度的增长。此外，生成式 AI 能够利用大量的车辆历史数据

[2] {'source': '道通24年年报', 'doc_id': '道通24年年报__full', 'section_title': '9、其他', 'section_id': '道通24年年报__full::section_583', 'source_doc_count': 306, 'doc_group': 'full_document', 'chunk_in_section': 0, 'content_level': 'chun

### Multi-Query

In [4]:
# 2) Multi-Query：围绕同一问题生成“不同标题锚点”的多路检索 query

multi_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "你是面向中文年报检索的 Multi-Query 生成器。"
            "给定一个问题，请先识别问题主题，再输出 4 条中文 query，每条一行，不要编号，不要解释。\n"
            "这 4 条 query 必须分别覆盖以下 4 种角度：\n"
            "1. 原问题压缩版：保留公司、年份、核心主题；\n"
            "2. 章节标题版：优先使用年报里可能真实出现的章节/小节标题；\n"
            "3. 子主题展开版：把该主题拆成 2 到 4 个最可能回答问题的子点；\n"
            "4. 关键词聚合版：把公司、年份、主题词、近义词和标题锚点组合成一个更像检索式的 query；\n"
            "如果问题是‘核心竞争力/竞争优势/技术壁垒/护城河/领先性’，优先围绕这些词生成：报告期内核心竞争力分析、核心竞争力分析、技术创新优势、产品及解决方案优势、全球化本地化营销服务体系优势、全球化本地化产能及供应链优势、人才与团队优势。\n"
            "要求：\n"
            "- 保留公司名和年份；\n"
            "- 4 条 query 必须明显不同，不能只是换同义词；\n"
            "- 不要编造数字；\n"
            "- 只输出 4 行 query。",
        ),
        ("human", "问题：{q}"),
    ]
)

queries = [
    s.strip()
    for s in llm.invoke(multi_prompt.format_messages(q=QUESTION)).content.splitlines()
    if s.strip()
]
print("queries:")
for q in queries:
    print("-", q)

# 融合策略：RRF（Reciprocal Rank Fusion）
rrf_k = 60
per_query_k = 8
rrf_scores = {}
doc_by_key = {}

for q in queries:
    docs = vs.similarity_search(q, k=per_query_k)
    for rank, d in enumerate(docs, 1):
        key = (d.metadata.get("chunk_id"), d.page_content[:80])
        doc_by_key[key] = d
        rrf_scores[key] = rrf_scores.get(key, 0.0) + 1.0 / (rrf_k + rank)

merged = [doc_by_key[key] for key in sorted(rrf_scores, key=rrf_scores.get, reverse=True)]

print("merged hits:", len(merged))
for i, d in enumerate(merged[:8], 1):
    print(f"[{i}]", d.metadata)
    print(d.page_content[:260].replace("\n", " "))
    print()


queries:
- 公司在海外市场的布局
- 海外市场布局
- 海外市场战略、海外销售网络、海外生产基地
- 公司 海外市场 布局 国际化 全球化 海外业务
merged hits: 19
[1] {'chunk_in_section': 0, 'h2': '四、财务报表的编制基础', 'source_doc_count': 306, 'page_start': 1, 'parse_source': 'paddleocr_vl', 'content_level': 'chunk', 'section_title': '2、持续经营', 'source': '道通24年年报', 'page_end': 306, 'file_path': 'paddleocr_vl/autel_annual_report_2024.md', 'h1': '母公司资产负债表', 'section_in_doc': 295, 'chunk_id': 394, 'doc_id': '道通24年年报__full', 'doc_group': 'full_document', 'h3': '2、持续经营', 'chunk_in_doc': 394, 'section_id': '道通24年年报__full::section_295'}
√适用 □不适用   本公司不存在导致对报告期末起 12 个月内的持续经营能力产生重大疑虑的事项或情况。

[2] {'section_id': '道通24年年报__full::section_297', 'chunk_in_doc': 396, 'content_level': 'chunk', 'source_doc_count': 306, 'h2': '五、重要会计政策及会计估计', 'chunk_id': 396, 'section_title': '1、遵循企业会计准则的声明', 'doc_id': '道通24年年报__full', 'chunk_in_section': 0, 'h1': '母公司资产负债表', 'page_start': 1, 'page_end': 306, 'section_in_doc': 297, 'file_path': 'paddleocr_vl/autel_annual_report_2024.md', 'doc_g

### HyDE

In [5]:
hyde_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "你是 HyDE 模块。请为用户问题写一段可能出现在年报中的‘假设答案’，用正式书面语，尽量包含可检索的关键词（业务、产品线、收入、分部等）。不要编造具体数值。",
        ),
        ("human", "问题：{q}"),
    ]
)

hypo = llm.invoke(hyde_prompt.format_messages(q=QUESTION)).content.strip()
print("hypo (first 400 chars):\n", hypo[:400])

hits = vs.similarity_search(hypo, k=5)
for i, d in enumerate(hits, 1):
    print(f"[{i}]", d.metadata)
    print(d.page_content[:260].replace("\n", " "))
    print()

hypo (first 400 chars):
 本集团持续深化全球化战略布局，积极拓展海外市场，以提升国际竞争力与市场份额。在美洲、欧洲、亚太等主要区域均设立了分支机构或运营中心，构建了覆盖广泛的销售与服务网络。核心业务板块，包括数字解决方案、高端装备制造及绿色能源产品线，已成功进入多个海外市场，并通过本地化运营与战略合作，适配不同区域的法规要求与市场需求。报告期内，国际业务分部收入贡献稳步增长，新兴市场的开拓取得积极进展，特别是在东南亚及中东地区的基础设施建设项目中，公司的技术产品与服务获得了重要应用。未来，集团将继续优化全球资源配置，加大在海外研发与产能方面的投入，以巩固和扩大在国际市场的领先地位。
[1] {'page_end': 306, 'parse_source': 'paddleocr_vl', 'content_level': 'chunk', 'page_start': 1, 'doc_id': '道通24年年报__full', 'chunk_id': 214, 'section_title': '(六) 有利于保护生态、防治污染、履行环境责任的相关信息', 'h1': '2024 年度报告', 'doc_group': 'full_document', 'source': '道通24年年报', 'file_path': 'paddleocr_vl/autel_annual_report_2024.md', 'source_doc_count': 306, 'h3': '(六) 有利于保护生态、防治污染、履行环境责任的相关信息', 'chunk_in_section': 0, 'h2': '4、公司环保管理制度等情况', 'chunk_in_doc': 214, 'section_in_doc': 164, 'section_id': '道通24年年报__full::section_164'}
##### ✓适用 ☐不适用   公司在材料循环回收利用与可持续发展方向，非常重视循环经济的实际推进与有效落地。2024年，公司筹划并发起“Evergreen”全球ESG植树活动，携手全球六大区域的权威公益组织，深度助力逾20家战略客户巩固其全球绿色品牌形象，充分展现出行业领导者的责任担当与影响力。   从业务实践看，道通科技在美国、越南和中国的生产基地采

### Parent-Child

**核心思想：** 检索需要细粒度（chunk 级别），但 LLM 生成需要大背景（section 级别）。

做法：
1. 用 **chunk-level** collection 做向量检索，命中最相关的 chunk
2. 从命中的 chunk metadata 里拿到 `section_id`
3. 用 `section_id` 去 **section-level** collection 查回完整的 section 文本
4. 把 section 文本交给 LLM，而不是只给小 chunk

这解决了 RAG 里经典的"检索精度 vs. 上下文完整性"矛盾。

In [6]:
# 4) Small-to-Big：在 chunk 上检索，返回 parent section 给 LLM

from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough


def small_to_big_retrieve(query: str, k: int = 5, verbose: bool = True):
    """在 chunk collection 上检索，再通过 section_id 回溯 parent section。"""
    chunk_hits = vs.similarity_search(query, k=k)

    section_ids_seen = set()
    parent_sections = []

    for hit in chunk_hits:
        sid = hit.metadata.get("section_id")
        if not sid or sid in section_ids_seen:
            continue
        section_ids_seen.add(sid)

        section_result = section_vs.get(ids=[sid])
        if section_result and section_result["documents"]:
            parent_sections.append({
                "section_id": sid,
                "section_title": section_result["metadatas"][0].get("section_title", ""),
                "text": section_result["documents"][0],
                "triggered_by_chunks": [
                    h.metadata.get("chunk_in_section")
                    for h in chunk_hits
                    if h.metadata.get("section_id") == sid
                ],
            })

    if verbose:
        print(f"query: {query}")
        print(f"chunk hits: {len(chunk_hits)} → unique parent sections: {len(parent_sections)}\n")
        for i, sec in enumerate(parent_sections, 1):
            print(f"[section {i}] {sec['section_id']}")
            print(f"  title: {sec['section_title']}")
            print(f"  triggered by chunk_in_section: {sec['triggered_by_chunks']}")
            print(f"  text preview: {sec['text'][:300].replace(chr(10), ' ')}")
            print()

    return parent_sections


parent_sections = small_to_big_retrieve(QUESTION, k=5)


query: 公司在海外市场有哪些布局？
chunk hits: 5 → unique parent sections: 5

[section 1] 道通24年年报__full::section_295
  title: 2、持续经营
  triggered by chunk_in_section: [0]
  text preview: √适用 □不适用   本公司不存在导致对报告期末起 12 个月内的持续经营能力产生重大疑虑的事项或情况。

[section 2] 道通24年年报__full::section_585
  title: 2、本企业的子公司情况
  triggered by chunk_in_section: [0]
  text preview: √适用 □不适用   本企业子公司的情况详见本附注十之说明。

[section 3] 道通24年年报__full::section_297
  title: 1、遵循企业会计准则的声明
  triggered by chunk_in_section: [0]
  text preview: 本公司所编制的财务报表符合企业会计准则的要求，真实、完整地反映了公司的财务状况、经营成果和现金流量等有关信息。

[section 4] 道通24年年报__full::section_605
  title: 1、重要承诺事项
  triggered by chunk_in_section: [0]
  text preview: ✓适用 ☐不适用   资产负债表日存在的对外重要承诺. 性质. 金额   已签订的正在或准备履行的租赁合同（不包括已确认使用权资产的租赁合同）及财务影响详见本财务报表附注七（82）2之说明。

[section 5] 道通24年年报__full::section_87
  title: 3、截至报告期末主要资产受限情况
  triggered by chunk_in_section: [0]
  text preview: ##### ✓适用 ☐不适用   期末银行存款中存在7,500.00元押金，3,892,370.84元冻结资金，其他货币资金中包含27,417,089.32元海关保证金、84,885,705.74元票据保证金、2,019,146.72元保函保证金、495,701.2

### 同题回答对比：Baseline / Rewrite / Multi-Query(RRF) / HyDE / Parent-Child

In [9]:
import pandas as pd
from IPython.display import display
from langchain_core.prompts import ChatPromptTemplate


answer_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "你是年报分析助手。基于给定检索内容回答问题。"
            "优先给出结构化结论；若证据不足，明确说明缺失信息。\n\n"
            "检索内容：\n{context}",
        ),
        ("human", "{question}"),
    ]
)


def get_doc_title(d):
    return d.metadata.get("section_title") or d.metadata.get("h3") or d.metadata.get("h2") or "无标题"


def build_context_from_docs(docs, max_docs=5, max_chars=1200):
    blocks = []
    for d in docs[:max_docs]:
        blocks.append(f"【{get_doc_title(d)}】\n{d.page_content[:max_chars]}")
    return "\n\n---\n\n".join(blocks)


def build_context_from_sections(sections, max_sections=5, max_chars=2000):
    return "\n\n---\n\n".join(
        f"【{sec['section_title']}】\n{sec['text'][:max_chars]}"
        for sec in sections[:max_sections]
    )


def answer_with_context(context):
    return llm.invoke(
        answer_prompt.format_messages(context=context, question=QUESTION)
    ).content.strip()


# 1) baseline
baseline_docs = vs.similarity_search(QUESTION, k=5)

# 2) rewrite
rewritten_q = llm.invoke(rewrite_prompt.format_messages(q=QUESTION)).content.strip()
rewrite_docs = vs.similarity_search(rewritten_q, k=5)

# 3) multi-query + RRF
multi_queries = [
    s.strip()
    for s in llm.invoke(multi_prompt.format_messages(q=QUESTION)).content.splitlines()
    if s.strip()
]
rrf_k = 60
per_query_k = 8
rrf_scores = {}
doc_by_key = {}
for q in multi_queries:
    docs = vs.similarity_search(q, k=per_query_k)
    for rank, d in enumerate(docs, 1):
        key = (d.metadata.get("chunk_id"), d.page_content[:80])
        doc_by_key[key] = d
        rrf_scores[key] = rrf_scores.get(key, 0.0) + 1.0 / (rrf_k + rank)
multi_docs = [doc_by_key[key] for key in sorted(rrf_scores, key=rrf_scores.get, reverse=True)][:5]

# 4) HyDE
hypo_q = llm.invoke(hyde_prompt.format_messages(q=QUESTION)).content.strip()
hyde_docs = vs.similarity_search(hypo_q, k=5)

# 5) Parent-Child (Small-to-Big)
parent_sections = small_to_big_retrieve(QUESTION, k=5, verbose=False)
parent_context = build_context_from_sections(parent_sections)

records = [
    {
        "strategy": "Baseline",
        "retrieval_query": QUESTION,
        "hits": len(baseline_docs),
        "hit_titles": " | ".join(get_doc_title(d) for d in baseline_docs),
        "answer": answer_with_context(build_context_from_docs(baseline_docs)),
    },
    {
        "strategy": "Rewrite",
        "retrieval_query": rewritten_q,
        "hits": len(rewrite_docs),
        "hit_titles": " | ".join(get_doc_title(d) for d in rewrite_docs),
        "answer": answer_with_context(build_context_from_docs(rewrite_docs)),
    },
    {
        "strategy": "Multi-Query (RRF)",
        "retrieval_query": " | ".join(multi_queries),
        "hits": len(multi_docs),
        "hit_titles": " | ".join(get_doc_title(d) for d in multi_docs),
        "answer": answer_with_context(build_context_from_docs(multi_docs)),
    },
    {
        "strategy": "HyDE",
        "retrieval_query": hypo_q[:2000].replace("\n", " ") + ("..." if len(hypo_q) > 2000 else ""),
        "hits": len(hyde_docs),
        "hit_titles": " | ".join(get_doc_title(d) for d in hyde_docs),
        "answer": answer_with_context(build_context_from_docs(hyde_docs)),
    },
    {
        "strategy": "Parent-Child",
        "retrieval_query": QUESTION,
        "hits": len(parent_sections),
        "hit_titles": " | ".join(sec["section_title"] for sec in parent_sections),
        "answer": answer_with_context(parent_context),
    },
]

comparison_df = pd.DataFrame(records)
comparison_df.insert(0, "question", QUESTION)

pd.set_option("display.max_colwidth", 180)
display(comparison_df)


,question,strategy,retrieval_query,hits,hit_titles,answer
0,公司在AI领域有哪些布局,Baseline,公司在AI领域有哪些布局,5,3、报告期内新技术、新产业、新业态、新模式的发展情况和未来发展趋势 | 9、其他 | 3、截至报告期末主要资产受限情况 | 4、持续和非持续第三层次公允价值计量项目，采用的估值技术和重要参数的定性及定量信息 | 1、公司负债情况,**结论：基于检索内容，公司未明确披露其在AI领域的具体业务布局或投资项目。**\n\n**分析依据：**\n\n1. **提及AI的关联性**：检索内容仅在“报告期内新技术、新产业、新业态、新模式的发展情况”部分，分析**行业趋势**时提及了生成式AI。具体描述为生成式AI能够赋能DaaS（诊断即服务）模式，革新汽车诊断和用户体验。**此描述为对...
1,公司在AI领域有哪些布局,Rewrite,公司在人工智能领域的布局,5,3、报告期内新技术、新产业、新业态、新模式的发展情况和未来发展趋势 | 9、其他 | 3、截至报告期末主要资产受限情况 | (1). 应收项目 | 4、持续和非持续第三层次公允价值计量项目，采用的估值技术和重要参数的定性及定量信息,**结论：基于现有信息，公司AI布局主要聚焦于汽车诊断领域，并已形成具体应用方向，但整体布局的广度和深度证据不足。**\n\n**结构化分析如下：**\n\n| 领域 | 具体布局/应用 | 证据来源 | 说明 |\n| :--- | :--- | :--- | :--- |\n| **汽车综合诊断及检测** | **生成式AI在DaaS（诊断即服务...
2,公司在AI领域有哪些布局,Multi-Query (RRF),公司在AI领域布局 | 人工智能相关业务与技术 | AI领域布局：技术研发、产品应用、战略合作、市场拓展 | 公司 AI 布局 人工智能 技术 产品 战略 年报 章节,5,3、计入当期损益的政府补助 | (2). 应付项目 | 3、报告期内新技术、新产业、新业态、新模式的发展情况和未来发展趋势 | 2024年：AI深度赋能，业务实现跨越式增长 | 8、其他,**结论：公司在AI领域有明确且已落地的布局，主要聚焦于汽车诊断和新能源（充电）两大主业，并成立了专门实体进行推进。**\n\n**结构化分析如下：**\n\n| 布局领域 | 具体举措/应用 | 证据来源 |\n| :--- | :--- | :--- |\n| **1. 汽车综合诊断及检测** | **生成式AI赋能DaaS（诊断即服务）**：利...
3,公司在AI领域有哪些布局,HyDE,本公司在人工智能领域持续深化战略布局，已将其确立为未来发展的核心驱动力之一。我们通过自主研发与战略合作相结合的方式，在多个业务板块和产品线中推进AI技术的融合与应用。 在**技术研发与基础设施**层面，公司设立了专注于人工智能的研发中心，持续投入于机器学习、自然语言处理、计算机视觉等基础技术的研究。我们致力于构建统一的AI开发平台与大数据处理能力，...,5,拥抱 AI，筑梦未来 | （三）空地一体集群智慧解决方案——开启 AI+机器人业务，第三发展曲线应运而生 | (一)核心竞争力分析 | (二) 公司发展战略 | 1、技术革命性迭代的风险,根据检索内容，道通科技在AI领域的布局清晰且全面，已形成以AI为核心驱动力的战略体系。以下是结构化总结：\n\n### 一、战略定位\n* **核心战略**：公司将AI定位为“当前及未来五年最核心的战略”，推行“全面AI化”。\n* **战略目标**：致力于成为**AI行业大模型商业化应用龙头企业**。\n\n### 二、具体业务布局\n1....
4,公司在AI领域有哪些布局,Parent-Child,公司在AI领域有哪些布局,5,3、报告期内新技术、新产业、新业态、新模式的发展情况和未来发展趋势 | 9、其他 | 3、截至报告期末主要资产受限情况 | 4、持续和非持续第三层次公允价值计量项目，采用的估值技术和重要参数的定性及定量信息 | 1、公司负债情况,根据提供的检索内容，**无法直接得出**公司在AI领域具体布局的明确结论。\n\n**分析如下：**\n\n1. **检索内容性质**：您提供的材料主要描述了**行业发展趋势**（如DaaS、生成式AI、AI Agent、物理AI）和公司的**部分财务及资产状况**，属于宏观背景和运营数据。\n2. **缺失关键信息**：报告中未包含“**公司主...
